# Prepare the experiment and reuse the R18 training configuration

In [1]:
# Mount Drive, retrieve the committed R50 model, and inspect the R18 run config.
# Verify the dataset and selected R18 checkpoint before preparing R50 training.
# This cell does not start training.

from google.colab import drive
drive.mount("/content/drive")

import json
import subprocess
from pathlib import Path

repo_dir = Path("/content/facial-visual-profile")
repo_url = "https://github.com/DeweyYip/facial-visual-profile.git"

if repo_dir.exists():
    subprocess.run(
        ["git", "-C", str(repo_dir), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

r50_path = repo_dir / "models/resnet50.py"
assert r50_path.is_file(), f"R50 model file is missing: {r50_path}"

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
r18_run = drive_root / "runs/r18_seed0"
config_path = r18_run / "config.json"
checkpoint_path = r18_run / "epoch_03.pt"

assert config_path.is_file(), f"Missing R18 config: {config_path}"
assert checkpoint_path.is_file(), f"Missing selected R18 checkpoint: {checkpoint_path}"

data_dir = drive_root / "data"
required_data = (
    "img_align_celeba.zip",
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
)
missing = [name for name in required_data if not (data_dir / name).is_file()]
assert not missing, f"Missing Drive data files: {missing}"

config = json.loads(config_path.read_text(encoding="utf-8"))

print("Repository commit:", subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"],
    text=True,
).strip())
print("R50 source:", r50_path)
print("R18 checkpoint available:", checkpoint_path.name)
print("Drive dataset files: all present")
print("R18 training configuration:")
for key in ("seed", "epochs", "batch_size", "learning_rate", "optimizer", "precision"):
    if key in config:
        print(f"  {key}: {config[key]}")
print("Full config keys:", sorted(config.keys()))
print("Preparation check passed. Training has not started.")

Mounted at /content/drive
Repository commit: f601fb6
R50 source: /content/facial-visual-profile/models/resnet50.py
R18 checkpoint available: epoch_03.pt
Drive dataset files: all present
R18 training configuration:
  seed: 0
  epochs: 10
  batch_size: 64
  learning_rate: 0.0001
  optimizer: Adam
  precision: FP32
Full config keys: ['attribute_names', 'augmentation', 'batch_size', 'cudnn_benchmark', 'cudnn_deterministic', 'environment', 'epoch_seed_rule', 'epochs', 'f1_threshold', 'git_commit', 'initial_weights', 'learning_rate', 'num_workers', 'optimizer', 'precision', 'seed', 'selection_metric', 'source_sha256', 'weight_decay']
Preparation check passed. Training has not started.


# Build a clean R50 workspace and keep the shared code identical to R18.

In [ ]:
# Build an R50 workspace using the verified shared source from the R18 run.
# Copy the new R50 model from the GitHub checkout.
# Keep the GitHub checkout available for recording the model's source commit.

import hashlib
from pathlib import Path

work_dir = Path("/content/r50_seed0_workspace")

# we want to make sure except the model, every other settings remain the same
shared_files = (
    "datasets/celeba.py",
    "datasets/preprocessing.py",
    "configs/attributes.yaml",
    "evaluation/attributes.py",
)

pending_files = {}
changed_files = []

for name in shared_files:
    saved_source = r18_run / "source" / name
    assert saved_source.is_file(), f"Missing saved source: {saved_source}"

    contents = saved_source.read_bytes()
    expected_hash = config["source_sha256"][name]

    assert hashlib.sha256(contents).hexdigest() == expected_hash, (
        f"Saved R18 source failed verification: {name}"
    )
    pending_files[name] = contents

    current_hash = hashlib.sha256(
        (repo_dir / name).read_bytes()
    ).hexdigest()
    if current_hash != expected_hash:
        changed_files.append(name)

pending_files["models/resnet50.py"] = (
    repo_dir / "models/resnet50.py"
).read_bytes()

# Check existing workspace files before writing.
for name, contents in pending_files.items():
    destination = work_dir / name
    assert (
        not destination.exists()
        or destination.read_bytes() == contents
    ), f"Existing workspace file differs: {destination}"

for name, contents in pending_files.items():
    destination = work_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        destination.write_bytes(contents)

for package in ("models", "datasets", "evaluation"):
    (work_dir / package / "__init__.py").touch()

print("GitHub files differing from the R18 snapshot:", changed_files)
print("Verified shared source files:", len(shared_files))
print("R50 workspace:", work_dir)
print("Workspace preparation passed.")

GitHub files differing from the R18 snapshot: ['datasets/celeba.py', 'evaluation/attributes.py']
Verified shared source files: 4
R50 workspace: /content/r50_seed0_workspace
Workspace preparation passed.


# Prepare the CelebA training and validation data.

In [ ]:
# Prepare training and validation data for the verified R50 workspace.
# Reuse existing Colab images when available; otherwise extract from the archive.
# Select the workspace as the import location for subsequent cells.

import os
import shutil
import sys
import time
import zipfile
from pathlib import Path

raw_dir = work_dir / "data/raw"
raw_dir.mkdir(parents=True, exist_ok=True)

for name in (
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
):
    print("Copying:", name)
    shutil.copy2(data_dir / name, raw_dir / name)

split_names = {"0": [], "1": []}
with (raw_dir / "list_eval_partition.txt").open(
    encoding="utf-8-sig"
) as handle:
    for line in handle:
        name, split = line.split()
        if split in split_names:
            split_names[split].append(name)

# 0 is for train, 1 is for validation
assert len(split_names["0"]) == 162770
assert len(split_names["1"]) == 19867

required_names = split_names["0"] + split_names["1"]
image_dir = raw_dir / "img_align_celeba"
existing_images = repo_dir / "data/raw/img_align_celeba"

# Reuse a complete local image directory without duplicating image files.
if not image_dir.exists() and existing_images.is_dir():
    if all((existing_images / name).is_file() for name in required_names):
        image_dir.symlink_to(existing_images, target_is_directory=True)
        print("Reusing existing local images:", existing_images)

missing_names = [
    name for name in required_names
    if not (image_dir / name).is_file()
]
started = time.perf_counter()

if missing_names:
    source_archive = data_dir / "img_align_celeba.zip"
    existing_archive = repo_dir / "data/raw/img_align_celeba.zip"
    local_archive = raw_dir / "img_align_celeba.zip"

    if (
        existing_archive.is_file()
        and existing_archive.stat().st_size == source_archive.stat().st_size
    ):
        archive_path = existing_archive
    else:
        if (
            not local_archive.is_file()
            or local_archive.stat().st_size != source_archive.stat().st_size
        ):
            print("Copying image archive...")
            shutil.copy2(source_archive, local_archive)
        archive_path = local_archive

    print("Extracting missing images:", len(missing_names))
    with zipfile.ZipFile(archive_path) as archive:
        for name in missing_names:
            archive.extract(f"img_align_celeba/{name}", path=raw_dir)

assert all((image_dir / name).is_file() for name in required_names)

# Detect earlier imports that would bypass the verified workspace.
for module_name in (
    "datasets",
    "models",
    "evaluation",
):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None)
    if module_file:
        assert Path(module_file).resolve().is_relative_to(
            work_dir.resolve()
        ), f"An earlier import conflicts with the workspace: {module_name}"

os.chdir(work_dir)
sys.path.insert(0, str(work_dir))

print("Training images ready:", len(split_names["0"]))
print("Validation images ready:", len(split_names["1"]))
print("Working directory:", Path.cwd())
print(f"Preparation time: {(time.perf_counter() - started) / 60:.1f} minutes")
print("Local data preparation passed.")

Copying: list_attr_celeba.txt
Copying: list_landmarks_align_celeba.txt
Copying: list_eval_partition.txt
Copying image archive...
Extracting missing images: 182637
Training images ready: 162770
Validation images ready: 19867
Working directory: /content/r50_seed0_workspace
Preparation time: 1.1 minutes
Local data preparation passed.


# Run a one-batch smoke test to verify forward, loss, backward, and optimization.

In [5]:
# Check R50 forward, backward, and one Adam update on a real batch of 64.
# Measure GPU memory with FP32 and the R18 optimizer settings.
# Discard the smoke-test model before initializing the formal training run.

import gc
import inspect
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from datasets.celeba import CelebAAttributes
from models.resnet50 import ResNet50Attributes

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."

for component in (CelebAAttributes, ResNet50Attributes):
    source_path = Path(inspect.getfile(component)).resolve()
    assert source_path.is_relative_to(work_dir.resolve()), (
        f"Imported code is outside the verified workspace: {source_path}"
    )

torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

train_dataset = CelebAAttributes(
    split="train",
    root=str(raw_dir),
)
assert len(train_dataset) == 162770
assert list(train_dataset.attribute_names) == config["attribute_names"]

check_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)
images, targets, _, names = next(iter(check_loader))

def check_r50(images, targets):
    random.seed(config["seed"])
    np.random.seed(config["seed"])
    torch.manual_seed(config["seed"])
    torch.cuda.manual_seed_all(config["seed"])

    check_model = ResNet50Attributes(pretrained=True).cuda()
    check_optimizer = torch.optim.Adam(
        check_model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    criterion = nn.BCEWithLogitsLoss()

    gpu_images = images.cuda(non_blocking=True)
    gpu_targets = targets.cuda(non_blocking=True)

    assert gpu_images.dtype == gpu_targets.dtype == torch.float32
    assert all(
        parameter.dtype == torch.float32
        for parameter in check_model.parameters()
    )

    check_model.train()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()

    check_optimizer.zero_grad(set_to_none=True)
    logits = check_model(gpu_images)
    loss = criterion(logits, gpu_targets)

    assert logits.shape == gpu_targets.shape == (64, 24)
    assert torch.isfinite(logits).all()
    assert torch.isfinite(loss)

    loss.backward()

    for name, parameter in check_model.named_parameters():
        if parameter.requires_grad:
            assert parameter.grad is not None, f"Missing gradient: {name}"
            assert torch.isfinite(parameter.grad).all(), (
                f"Nonfinite gradient: {name}"
            )

    backbone_norm = check_model.network.conv1.weight.grad.norm().item()
    head_norm = check_model.network.fc.weight.grad.norm().item()
    assert backbone_norm > 0 and head_norm > 0

    check_optimizer.step()

    assert all(
        torch.isfinite(parameter).all()
        for parameter in check_model.parameters()
    )

    torch.cuda.synchronize()
    return {
        "gpu": torch.cuda.get_device_name(0),
        "torch": str(torch.__version__),
        "batch_size": len(images),
        "precision": "FP32",
        "logits_shape": list(logits.shape),
        "loss": loss.item(),
        "backbone_gradient_norm": backbone_norm,
        "head_gradient_norm": head_norm,
        "peak_allocated_gib": (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        "elapsed_seconds": time.perf_counter() - started,
        "passed": True,
    }

smoke_report = check_r50(images, targets)

gc.collect()
torch.cuda.empty_cache()

r50_run = drive_root / "runs/r50_seed0"
r50_run.mkdir(parents=True, exist_ok=True)
report_path = r50_run / "gpu_smoke.json"
report_path.write_text(
    json.dumps(smoke_report, indent=2) + "\n",
    encoding="utf-8",
)

print("GPU:", smoke_report["gpu"])
print("Batch size:", smoke_report["batch_size"])
print("Logits:", tuple(smoke_report["logits_shape"]))
print(f"Smoke-test BCE loss: {smoke_report['loss']:.6f}")
print(f"Backbone gradient norm: {smoke_report['backbone_gradient_norm']:.6f}")
print(f"Head gradient norm: {smoke_report['head_gradient_norm']:.6f}")
print(f"Peak allocated GPU memory: {smoke_report['peak_allocated_gib']:.2f} GiB")
print("Forward, backward, and Adam update: PASS")
print("Saved:", report_path)
print("Formal training has not started.")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 166MB/s]


GPU: Tesla T4
Batch size: 64
Logits: (64, 24)
Smoke-test BCE loss: 0.704011
Backbone gradient norm: 0.070611
Head gradient norm: 1.478321
Peak allocated GPU memory: 5.30 GiB
Forward, backward, and Adam update: PASS
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0/gpu_smoke.json
Formal training has not started.


# Initialize the formal R50 model and save the starting checkpoint.

In [6]:
# Initialize the formal R50 seed-0 run using the R18 training settings.
# Save the actual source files, configuration, and initial checkpoint.
# Verify model reload and optimizer settings before starting epoch 1.

import copy
import hashlib
import importlib.metadata
import platform
import subprocess

valid_dataset = CelebAAttributes(
    split="valid",
    root=str(raw_dir),
)
assert len(valid_dataset) == 19867
assert list(valid_dataset.attribute_names) == config["attribute_names"]

source_names = (
    "models/resnet50.py",
    "datasets/celeba.py",
    "datasets/preprocessing.py",
    "configs/attributes.yaml",
    "evaluation/attributes.py",
)

source_contents = {
    name: (work_dir / name).read_bytes()
    for name in source_names
}

r50_config = copy.deepcopy(config)
r50_config.update({
    "model": "ResNet50Attributes",
    "initial_weights": "ResNet50_Weights.IMAGENET1K_V1",
    "git_commit": subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
        text=True,
    ).strip(),
    "shared_source_reference": "runs/r18_seed0/source",
    "source_sha256": {
        name: hashlib.sha256(contents).hexdigest()
        for name, contents in source_contents.items()
    },
    "environment": {
        "python": platform.python_version(),
        "torch": str(torch.__version__),
        "torchvision": importlib.metadata.version("torchvision"),
        "numpy": np.__version__,
        "Pillow": importlib.metadata.version("Pillow"),
        "scikit-learn": importlib.metadata.version("scikit-learn"),
        "gpu": torch.cuda.get_device_name(0),
    },
})

# Keep all shared training settings equal to the R18 configuration.
for key in (
    "seed", "epochs", "batch_size", "learning_rate",
    "weight_decay", "optimizer", "precision", "augmentation",
    "f1_threshold", "selection_metric", "epoch_seed_rule",
    "cudnn_deterministic", "cudnn_benchmark", "num_workers",
):
    assert r50_config[key] == config[key], f"Setting differs: {key}"

config_path = r50_run / "config.json"
initial_path = r50_run / "epoch_00.pt"

if config_path.exists():
    stored_config = json.loads(config_path.read_text(encoding="utf-8"))
    assert stored_config == r50_config, (
        "An existing R50 run has a different configuration."
    )

for name, contents in source_contents.items():
    destination = r50_run / "source" / name
    assert (
        not destination.exists()
        or destination.read_bytes() == contents
    ), f"Existing saved source differs: {name}"

for name, contents in source_contents.items():
    destination = r50_run / "source" / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        destination.write_bytes(contents)

if not config_path.exists():
    temporary_config = r50_run / "config.json.tmp"
    temporary_config.write_text(
        json.dumps(r50_config, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_config.replace(config_path)

random.seed(r50_config["seed"])
np.random.seed(r50_config["seed"])
torch.manual_seed(r50_config["seed"])
torch.cuda.manual_seed_all(r50_config["seed"])

device = torch.device("cuda")
model = ResNet50Attributes(pretrained=True).to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=r50_config["learning_rate"],
    weight_decay=r50_config["weight_decay"],
)
criterion = nn.BCEWithLogitsLoss()

if not initial_path.exists():
    temporary_checkpoint = r50_run / "epoch_00.pt.tmp"
    torch.save({
        "epoch": 0,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "config": r50_config,
        "history": [],
    }, temporary_checkpoint)
    temporary_checkpoint.replace(initial_path)

initial_checkpoint = torch.load(
    initial_path,
    map_location="cpu",
    weights_only=True,
)
assert initial_checkpoint["epoch"] == 0
assert initial_checkpoint["config"] == r50_config

# Compare freshly initialized predictions with the saved and reloaded model.
model.eval()
probe_images = images[:4].to(device)

with torch.inference_mode():
    expected_logits = model(probe_images).cpu()

model.load_state_dict(initial_checkpoint["model_state"], strict=True)
optimizer.load_state_dict(initial_checkpoint["optimizer_state"])

with torch.inference_mode():
    restored_logits = model(probe_images).cpu()

torch.testing.assert_close(
    restored_logits,
    expected_logits,
    rtol=0,
    atol=0,
)
assert optimizer.param_groups[0]["lr"] == r50_config["learning_rate"]
assert optimizer.param_groups[0]["weight_decay"] == r50_config["weight_decay"]

del initial_checkpoint, probe_images
gc.collect()
torch.cuda.empty_cache()

print("Training images:", len(train_dataset))
print("Validation images:", len(valid_dataset))
print("Shared training settings: matched")
print("Initial checkpoint reload: PASS")
print("Seed:", r50_config["seed"])
print("Epochs:", r50_config["epochs"])
print("Saved config:", config_path)
print("Saved source:", r50_run / "source")
print("Saved initial checkpoint:", initial_path)
print("Formal training has not started.")

Training images: 162770
Validation images: 19867
Shared training settings: matched
Initial checkpoint reload: PASS
Seed: 0
Epochs: 10
Saved config: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0/config.json
Saved source: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0/source
Saved initial checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0/epoch_00.pt
Formal training has not started.


Formal training

In [ ]:
# Train R50 seed 0 for 10 epochs with the established R18 protocol.
# Save a complete checkpoint after every epoch and select by validation mAP.
# Resume from the latest completed epoch when this cell is rerun.
# Verify model and Adam-state restoration after the first epoch.

import math
import shutil
import time

from tqdm.auto import tqdm
from evaluation.attributes import evaluate_attributes

assert Path(inspect.getfile(evaluate_attributes)).resolve().is_relative_to(
    work_dir.resolve()
)

# make sure we use the same randomness every time so each epoch is reproducible
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# give each DataLoader worker a reproducible random seed as well
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# safely write JSON files
def write_json_atomic(path, contents):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(contents, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

# we save everything in case it process breaks up. so we can restore the process
# literally everything is saved: from model_state to optimizer state 
def save_checkpoint(epoch, history):
    filename = f"epoch_{epoch:02d}.pt"
    local_path = work_dir / "outputs" / filename
    local_path.parent.mkdir(parents=True, exist_ok=True)

    # save everything
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "config": r50_config,
        "history": history,
    }, local_path)

    destination = r50_run / filename
    temporary = r50_run / (filename + ".tmp")

    # store to the local first and then drive
    shutil.copy2(local_path, temporary)
    temporary.replace(destination)
    local_path.unlink()
    return destination

# only run one time after epoch 1 
# # Verify that the epoch-1 checkpoint can fully restore both the model and Adam optimizer.
def verify_epoch_one(path, probe_images):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    assert saved["epoch"] == 1
    assert saved["config"] == r50_config
    assert len(saved["history"]) == 1

    restored_model = ResNet50Attributes(pretrained=False).to(device)
    restored_model.load_state_dict(saved["model_state"], strict=True)
    restored_optimizer = torch.optim.Adam(
        restored_model.parameters(),
        lr=r50_config["learning_rate"],
        weight_decay=r50_config["weight_decay"],
    )
    restored_optimizer.load_state_dict(saved["optimizer_state"])

    expected_steps = math.ceil(
        len(train_dataset) / r50_config["batch_size"]
    )
    assert len(restored_optimizer.state) == len(optimizer.state)
    assert len(restored_optimizer.state) > 0

    for state in restored_optimizer.state.values():
        assert int(state["step"].item()) == expected_steps
        assert torch.isfinite(state["exp_avg"]).all()
        assert torch.isfinite(state["exp_avg_sq"]).all()

    restored_model.eval()
    model.eval()
    with torch.inference_mode():
        expected = model(probe_images)
        actual = restored_model(probe_images)
    torch.testing.assert_close(actual, expected, rtol=0, atol=0)

    write_json_atomic(r50_run / "checkpoint_restore_check.json", {
        "checkpoint": path.name,
        "model_reload_passed": True,
        "optimizer_reload_passed": True,
        "optimizer_steps": expected_steps,
    })
    print("Epoch-1 model and Adam restoration: PASS", flush=True)

# Restore the latest complete checkpoint, including epoch 0.
checkpoint_paths = sorted(r50_run.glob("epoch_[0-9][0-9].pt"))
assert checkpoint_paths, "No initial checkpoint was found."
latest_path = checkpoint_paths[-1]
saved = torch.load(latest_path, map_location="cpu", weights_only=True)

# restoration process
assert saved["config"] == r50_config, "Checkpoint configuration differs."
model.load_state_dict(saved["model_state"], strict=True)
# restore Adam
optimizer.load_state_dict(saved["optimizer_state"])
# restore history
history = saved["history"]
# restore preocess
completed_epoch = int(saved["epoch"])

# check if everything's fine
assert len(history) == completed_epoch
assert [record["epoch"] for record in history] == list(
    range(1, completed_epoch + 1)
)
assert completed_epoch <= r50_config["epochs"]
del saved
gc.collect()
torch.cuda.empty_cache()
print("Loaded:", latest_path.name, flush=True)
print("Completed epochs:", completed_epoch, flush=True)
print("Next epoch:", completed_epoch + 1, flush=True)

# the official epoch loop
"""

"""

for epoch in range(completed_epoch + 1, r50_config["epochs"] + 1):

    # Match the R18 epoch-specific seeding and shuffle protocol.
    epoch_seed = r50_config["seed"] + epoch
    seed_everything(epoch_seed)
    generator = torch.Generator().manual_seed(epoch_seed)

    # we shuffle everything inside
    train_loader = DataLoader(
        train_dataset,
        batch_size=r50_config["batch_size"],
        shuffle=True,
        num_workers=r50_config["num_workers"],
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=generator,
    )
    # we don't shuffle validation batches 
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=r50_config["batch_size"],
        shuffle=False,
        num_workers=r50_config["num_workers"],
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
    )

    started = time.perf_counter()
    # train mode on babe ~
    model.train()
    # for calculating average loss
    train_loss_sum = 0.0
    train_samples = 0

    # For each batch:
    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{r50_config['epochs']} train",
    )
    # we only need images and tragets out of all 4 outputs from progress
    for images, targets, _, _ in progress:

        # put the data into our device
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        # 1. optimizer zero grad 
        optimizer.zero_grad(set_to_none=True)

        # 2. forward pass
        logits = model(images)

        # 3. calulate the loss loss loss
        loss = criterion(logits, targets)
        assert torch.isfinite(loss), "Nonfinite training loss."

        # 4. loss backward
        loss.backward()

        # 5. optimizer step step step
        optimizer.step()

        # update the loss of this batch
        # accumulate the loss for epoch stats
        # loss.item() represents the avg loss of this batch
        batch_size = len(images)
        train_loss_sum += loss.item() * batch_size
        train_samples += batch_size
        # to get the average training loss
        progress.set_postfix(loss=f"{train_loss_sum / train_samples:.4f}")

    assert train_samples == len(train_dataset)

    # evaluation mode on babe ~
    model.eval()
    valid_loss_sum = 0.0
    valid_samples = 0
    validation_logits = []
    validation_targets = []

    with torch.inference_mode():
        for images, targets, _, _ in tqdm(
            valid_loader,
            desc=f"Epoch {epoch}/{r50_config['epochs']} valid",
        ):
            
            # put data to the device first
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            # forward pass 
            logits = model(images)
            loss = criterion(logits, targets)
            assert torch.isfinite(logits).all()
            assert torch.isfinite(loss)

            # loss.iitem() is the avg loss of this batch
            batch_size = len(images)
            valid_loss_sum += loss.item() * batch_size
            valid_samples += batch_size

            # save the results
            validation_logits.append(logits.cpu().numpy())
            validation_targets.append(targets.cpu().numpy())

    assert valid_samples == len(valid_dataset)


    # put all batched together 
    # validation is the different stat results of the 19867 samples
    validation = evaluate_attributes(
        np.concatenate(validation_logits),
        np.concatenate(validation_targets),
        threshold=r50_config["f1_threshold"],
    )
    record = {
        "epoch": epoch,
        "train_loss": train_loss_sum / train_samples,
        "valid_loss": valid_loss_sum / valid_samples,
        "validation": validation,
        "seconds_excluding_save": time.perf_counter() - started,
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {record['train_loss']:.6f} | "
        f"valid loss {record['valid_loss']:.6f} | "
        f"macro F1 {validation['macro_f1']:.6f} | "
        f"mAP {validation['map']:.6f} | "
        f"time {record['seconds_excluding_save'] / 60:.1f} min",
        flush=True,
    )

    # save checkpoint and history  
    checkpoint_path = save_checkpoint(epoch, history)
    write_json_atomic(r50_run / "history.json", history)

    # find the best epoch by comparing mAP
    best_record = max(
        history,
        key=lambda item: item["validation"]["map"],
    )
    # write json to record the info about the best epoch
    write_json_atomic(r50_run / "best.json", {
        "epoch": best_record["epoch"],
        "checkpoint": f"epoch_{best_record['epoch']:02d}.pt",
        "selection_metric": r50_config["selection_metric"],
        "validation": best_record["validation"],
        "valid_loss": best_record["valid_loss"],
    })

    if epoch == 1:
        verify_epoch_one(checkpoint_path, images[:4])
        gc.collect()
        torch.cuda.empty_cache()

    print("Saved checkpoint:", checkpoint_path, flush=True)
    print(
        f"Best completed epoch: {best_record['epoch']} | "
        f"mAP: {best_record['validation']['map']:.6f}",
        flush=True,
    )

print("All planned epochs completed.", flush=True)

Loaded: epoch_00.pt
Completed epochs: 0
Next epoch: 1


Epoch 1/10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

Epoch 1/10 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 01 | train loss 0.210346 | valid loss 0.194427 | macro F1 0.731537 | mAP 0.814506 | time 28.5 min
Epoch-1 model and Adam restoration: PASS
Saved checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0/epoch_01.pt
Best completed epoch: 1 | mAP: 0.814506


Epoch 2/10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

...

Epoch 05 | train loss 0.134019 | valid loss 0.213929 | macro F1 0.735291 | mAP 0.804058 | time 28.6 min
Saved checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0/epoch_05.pt
Best completed epoch: 3 | mAP: 0.818835


Epoch 6/10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

Epoch 6/10 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 06 | train loss 0.109007 | valid loss 0.240889 | macro F1 0.735797 | mAP 0.795161 | time 28.6 min
Saved checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0/epoch_06.pt
Best completed epoch: 3 | mAP: 0.818835


Epoch 7/10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

In [1]:
# Mount Google Drive and inspect the latest saved R50 checkpoint.
from pathlib import Path
from google.colab import drive
import torch

drive.mount("/content/drive")

run_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/r50_seed0"
)
assert run_dir.is_dir(), f"Run directory not found: {run_dir}"

checkpoints = sorted(run_dir.glob("epoch_[0-9][0-9].pt"))
assert checkpoints, "No saved checkpoints found."

for path in checkpoints:
    print(f"{path.name}: {path.stat().st_size / 1024**2:.2f} MiB")

latest = checkpoints[-1]
checkpoint = torch.load(
    latest, map_location="cpu", weights_only=True
)

completed_epoch = int(checkpoint["epoch"])
planned_epochs = int(checkpoint["config"]["epochs"])

print("\nLatest readable checkpoint:", latest.name)
print("Completed epoch:", completed_epoch)
print("Next epoch:", completed_epoch + 1)
print("Planned epochs:", planned_epochs)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

del checkpoint

Mounted at /content/drive
epoch_00.pt: 90.17 MiB
epoch_01.pt: 270.03 MiB
epoch_02.pt: 270.03 MiB
epoch_03.pt: 270.03 MiB
epoch_04.pt: 270.03 MiB
epoch_05.pt: 270.03 MiB
epoch_06.pt: 270.03 MiB
epoch_07.pt: 270.04 MiB
epoch_08.pt: 270.04 MiB

Latest readable checkpoint: epoch_08.pt
Completed epoch: 8
Next epoch: 9
Planned epochs: 10
CUDA available: False
